In [43]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np 
from pathlib import Path, WindowsPath
import matplotlib.pyplot as plt
import tensorflow as tf
import keras 
from keras import backend as K
from tensorflow.keras.models import Sequential
import tensorflow.keras.layers as layers
from tensorflow import data as tf_data
from Custom_Modules import PreProcessing as PreProcess
from tensorflow.keras import optimizers, losses

K.set_floatx('float32')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
Current_Dir = Path.cwd()
Root_Dir = Current_Dir.parent
Data_Dir = Root_Dir / "Data" / "mfr2"


In [4]:
train_df = pd.read_csv(Data_Dir/'simplified_labels.txt')

In [5]:
train_df

,person_name,image_num,mask_status
0,AdrianDunbar,1,mask
1,AdrianDunbar,2,no-mask
2,AdrianDunbar,3,mask
3,AdrianDunbar,4,no-mask
4,AhmadAli,1,mask
...,...,...,...
264,ZuzanaCaputova,4,mask
265,ZuzanaCaputova,5,no-mask
266,ZuzanaCaputova,6,mask
267,ZuzanaCaputova,7,no-mask


In [25]:

filename = {PreProcess.compile_file_name(line['person_name'], line['image_num']): PreProcess.mask_encoding(line['mask_status']) for index,line in train_df.iterrows()}


In [26]:
filename

{'AdrianDunbar_0001.png': 1,
 'AdrianDunbar_0002.png': 0,
 'AdrianDunbar_0003.png': 1,
 'AdrianDunbar_0004.png': 0,
 'AhmadAli_0001.png': 1,
 'AhmadAli_0002.png': 0,
 'AhmadAli_0003.png': 1,
 'AhmadAli_0004.png': 1,
 'AhmadAli_0005.png': 0,
 'ArifAlvi_0001.png': 1,
 'ArifAlvi_0002.png': 1,
 'ArifAlvi_0003.png': 0,
 'ArifAlvi_0004.png': 0,
 'BellaHadid_0001.png': 1,
 'BellaHadid_0002.png': 0,
 'BellaHadid_0003.png': 1,
 'BenAffleck_0001.png': 1,
 'BenAffleck_0002.png': 1,
 'BenAffleck_0003.png': 0,
 'BenAffleck_0004.png': 1,
 'BenAffleck_0005.png': 1,
 'BenAffleck_0006.png': 1,
 'BenAffleck_0007.png': 0,
 'BrianKemp_0001.png': 0,
 'BrianKemp_0002.png': 1,
 'BrianKemp_0003.png': 1,
 'BrianKemp_0004.png': 1,
 'BrianKemp_0005.png': 0,
 'BrodyJenner_0001.png': 0,
 'BrodyJenner_0002.png': 1,
 'CarrieLam_0001.png': 1,
 'CarrieLam_0002.png': 1,
 'CarrieLam_0003.png': 1,
 'CarrieLam_0004.png': 0,
 'CarrieLam_0005.png': 1,
 'CarrieLam_0006.png': 0,
 'CharlieBaker_0001.png': 0,
 'CharlieBaker_000

In [28]:
filename = [PreProcess.mask_encoding(line['mask_status']) for index,line in train_df.iterrows()]

In [38]:
train_dataset, test_dataset = keras.utils.image_dataset_from_directory(Data_Dir, labels = filename, label_mode= 'binary', image_size= (160,160), seed = 42, validation_split= .3, subset = 'both')

Found 269 files belonging to 2 classes.
Using 189 files for training.
Using 80 files for validation.


In [ ]:
train_dataset = train_dataset.map( #put augement function
    num_parallel_calls= tf_data.AUTOTUNE)

train_dataset = train_dataset.prefetch(tf_data.AUTOTUNE)
test_dataset = test_dataset.prefetch(tf_data.AUTOTUNE)


TypeError: DatasetV2.map() missing 1 required positional argument: 'map_func'

In [49]:
model = PreProcess.build_cnn(
    input_shape = (160,160,3), 
    num_class = 1
)

model.summary()

Model: "mfr2_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 160, 160, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation_1            │ (None, 160, 160, 3)    │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 160, 160, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 160, 160, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 80, 80, 128)    │        36,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 80, 80, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 64,417 (251.63 KB)

 Trainable params: 64,097 (250.38 KB)

 Non-trainable params: 320 (1.25 KB)

In [50]:
model.compile( 
    optimizer=optimizers.Adam(),    
    loss = losses.binary_crossentropy,
    metrics = ['accuracy']
)

In [51]:
model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=20
)

Epoch 1/20


c:\Users\Evan\anaconda3\envs\SpringboardCapstone3\Lib\site-packages\keras\src\ops\nn.py:959: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 416ms/step - accuracy: 0.6455 - loss: 0.7047 - val_accuracy: 0.6125 - val_loss: 0.6876
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 406ms/step - accuracy: 0.6455 - loss: 0.6446 - val_accuracy: 0.6125 - val_loss: 0.6816
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 390ms/step - accuracy: 0.6455 - loss: 0.6244 - val_accuracy: 0.6125 - val_loss: 0.6796
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 390ms/step - accuracy: 0.6455 - loss: 0.5987 - val_accuracy: 0.6125 - val_loss: 0.6802
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 380ms/step - accuracy: 0.6455 - loss: 0.5730 - val_accuracy: 0.6125 - val_loss: 0.6783
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 376ms/step - accuracy: 0.6455 - loss: 0.5685 - val_accuracy: 0.6125 - val_loss: 0.6772
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 378ms/step - accuracy: 0.6455 - loss: 0.5696 - val_accuracy: 0.6125 - val_loss: 0.6741
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 387ms/step - accuracy: 0.6455 - loss: 0.5974 - val_accuracy: 0.6125 - val_loss: 0.6735
Epo